In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [3]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="NandemoGHS/Japanese-Eroge-Voice", 
    repo_type="dataset", local_dir="./Japanese-Eroge-Voice", allow_patterns="*.tar")

Fetching 218 files: 100%|██████████| 218/218 [01:07<00:00,  3.21it/s]


'/home/ubuntu/Japanese-Eroge-Voice'

In [6]:
def loop(files):
    files, _ = files
    import tarfile

    for f in tqdm(files):
        with tarfile.open(f, "r") as tar:
            tar.extractall(path='Japanese-Eroge-Voice')
        os.remove(f)

In [9]:
multiprocessing(files, loop, cores = 20, returned = False)

100%|██████████| 10/10 [00:12<00:00,  1.25s/it]


In [30]:
files = glob('Japanese-Eroge-Voice/*.txt')
len(files)

222646

In [14]:
with open(files[0]) as fopen:
    d = fopen.read()
d

'戻しときましょ。それともあんた、食べてみたいの?\n'

In [31]:
def loop(files):
    files, _ = files
    data = []
    for f in tqdm(files):
        try:
            with open(f) as fopen:
                t = fopen.read().strip()
            if len(t) < 2:
                continue
    
            base = f.split('/')[0] + '_audio'
            f_audio = f.replace('.txt', '.flac')
            f_new = f_audio.replace('/', '-').replace('.parquet', '').replace('.flac', '')
            os.makedirs(base, exist_ok=True)
    
            audio_filename = f'{f_new}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            audio_np, sr = sf.read(f_audio)
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        except:
            pass
    return data

In [32]:
rows = loop((files[:10], 0))
rows[0]

100%|██████████| 10/10 [00:00<00:00, 10.06it/s]


{'audio_filename': 'Japanese-Eroge-Voice_audio/Japanese-Eroge-Voice-ae1422b120c2d902.mp3',
 'text': '戻しときましょ。それともあんた、食べてみたいの?',
 'speaker': 'Japanese-Eroge-Voice_audio'}

In [33]:
data = multiprocessing(files, loop, cores = 30)

100%|██████████| 7421/7421 [12:34<00:00,  9.84it/s]


In [34]:
with open('Japanese-Eroge-Voice.json', 'w') as fopen:
    json.dump(data, fopen)

In [26]:
audio_files = [d['audio_filename'] for d in data]

with open('Japanese-Eroge-Voice-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [37]:
# !zip -rq Japanese-Eroge-Voice_audio.zip Japanese-Eroge-Voice_audio

In [38]:
# !hf upload malaysia-ai/Multilingual-TTS Japanese-Eroge-Voice_audio.zip --repo-type=dataset